In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline

In [ ]:
appartments = pd.read_csv("data/appartments_train.csv")
appartments_test = pd.read_csv("data/appartments_test.csv")

#Floors over 4 were low in frequency
appartments["floor_max"] = appartments["floor_max"].fillna(4.0)
appartments_test["floor_max"] = appartments_test["floor_max"].fillna(4.0)

#Distances and others had missing values
for i in ["dist_sch","dist_clinic","dist_post","dist_kind","dist_rest","dist_uni","dist_pharma"]:
    appartments[i] = appartments[i].fillna(np.mean(appartments[i]))
for i in ["dist_sch","dist_clinic","dist_post","dist_kind","dist_rest","dist_uni","dist_pharma"]:
    appartments_test[i] = appartments_test[i].fillna(np.mean(appartments[i]))
    
appartments['cond_class'] = appartments['cond_class'].fillna("Missing")
appartments_test['cond_class'] = appartments_test['cond_class'].fillna("Missing")

appartments['has_lift'] = appartments['has_lift'].fillna('no')
appartments_test['has_lift'] = appartments_test['has_lift'].fillna('no')

appartments['obj_type'] = appartments['obj_type'].fillna('other')
appartments_test['obj_type'] = appartments_test['obj_type'].fillna('other')

appartments['build_mat'] = appartments['build_mat'].fillna('other')
appartments_test['build_mat'] = appartments_test['build_mat'].fillna('other')

#Didnt want to fill in with the year as it may lead to some non-sensical observations like a very old house build with modern materials so instead we used
#other potenitally related variables
appartments['year_built'] = appartments.groupby(['loc_code','obj_type'])['year_built'].transform(
    lambda x: x.fillna(x.median())).astype(int)
#Along the same resoning as above
appartments['infrastructure_quality'] = appartments.groupby('year_built')['infrastructure_quality'].transform(
    lambda x: x.fillna(x.mean()))

floor_medians = appartments.groupby('floor_max')['floor_no'].median()
appartments['floor_no'] = appartments['floor_no'].fillna(appartments['floor_max'].map(floor_medians))

appartments_test['year_built'] = appartments_test.groupby(['loc_code','obj_type'])['year_built'].transform(
    lambda x: x.fillna(x.median())).astype(int)
appartments_test['infrastructure_quality'] = appartments_test.groupby('year_built')['infrastructure_quality'].transform(
    lambda x: x.fillna(x.mean()))

floor_medians = appartments_test.groupby('floor_max')['floor_no'].median()
appartments_test['floor_no'] = appartments_test['floor_no'].fillna(appartments_test['floor_max'].map(floor_medians))

appartments.drop('dist_uni',axis=1, inplace = True)
appartments_test.drop('dist_uni',axis=1, inplace = True)

#Decoding into 0,1 for modelling
for i in ["has_park", "has_balcony", "has_lift", "has_sec", "has_store"]:
    appartments[i] = appartments[i].map({'no': 0, 'yes': 1})
for i in ["has_park", "has_balcony", "has_lift", "has_sec", "has_store"]:
    appartments_test[i] = appartments_test[i].map({'no': 0, 'yes': 1})

nominal_variables_enc = ['obj_type','own_type','build_mat','cond_class','loc_code','src_month']

#Decoding categorical variables into dummies
appartments = pd.get_dummies(appartments,
                                      columns = nominal_variables_enc,
                                      drop_first = True,
                                      dtype = int) # bool by default

appartments_test = pd.get_dummies(appartments_test,
                                     columns = nominal_variables_enc,
                                     drop_first = True,
                                     dtype = int)
appartments.drop('unit_id',axis=1, inplace = True)

# The part below takes 10-20min to run depending on PC specs. The score for LASSO and ridge was very similar with linear regression slighty behind and SVR
# scoring quite poorly it was also very computationaly expensive. We also choose LASSO because it can handle correlated coeffficients and prevents overfitting
# which is easy to run into when optimizing for a specific value like RMSE.
appartments_X = appartments.drop(['price_z'], axis = 1)
appartments_y = appartments['price_z']

alphas = np.logspace(3, -4, 50)

lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),  
    ('model', Lasso())  
])

cv5 = KFold(n_splits = 5,
            shuffle = True,
            random_state = 123)



param_grid = {'model__alpha': alphas}


lasso_search = GridSearchCV(
    estimator = lasso_pipeline,
    param_grid = param_grid,
    scoring = 'neg_mean_squared_error',
    cv = cv5,
    n_jobs = -1)  


lasso_search.fit(appartments_X, appartments_y)


print("Best alpha:", lasso_search.best_params_)
best_rmse = np.sqrt(-lasso_search.best_score_)
print("Best cross-validated RMSE:", best_rmse)

unit_ids = appartments_test['unit_id']
X_test = appartments_test.drop(columns=['unit_id'])

predicted_prices = lasso_search.best_estimator_.predict(X_test)

output_df = pd.DataFrame({
    'unit_id': unit_ids,
    'predicted_price_z': predicted_prices
})

output_df.to_csv("outputs/predicted_prices.csv", index=False)